In [113]:
%pip install numpy matplotlib meshio pygmsh pyvista anywidget 

Note: you may need to restart the kernel to use updated packages.


In [114]:
import pygmsh
import numpy as np
import matplotlib.pyplot as plt
import json
import meshio

# 設定
OUTPUT_JSON_FILE = 'input.json'
GMSH_OUTPUT_FILE_2D = 'model_mesh_2d.msh'
GMSH_OUTPUT_FILE_3D = 'model_mesh_3d.msh'

print("✅ Libraries loaded.")

✅ Libraries loaded.


In [115]:
# def generate_system_prompt():
#     # pygmsh.occ の正確なシグネチャを定義
#     reference_rules = """
# ### pygmsh.occ Code Generation Rules:
# 1. Geometry object: Use `geom` (already defined).
# 2. Rectangle: `rect = geom.add_rectangle([x0, y0, z0], width, height)`
# 3. Disk (Circle): `disk = geom.add_disk([xc, yc, zc], radius)`
# 4. Boolean Difference (A minus B): `geom.boolean_difference(A, B)`
# 5. Boolean Union (A plus B): `geom.boolean_union([list_of_entities])`
# 6. Precision: Always use float values (e.g., 100.0, not 100).
# 7. Output: Return ONLY pure Python code. No markdown code blocks, no explanations.
#     """

#     system_instruction = f"""
# You are an expert CAE Engineer specialized in pygmsh. 
# Create geometry definition code based on the user's vague request.
# {reference_rules}

# Example Request: "A square plate with a hole in the middle"
# Example Output:
# plate = geom.add_rectangle([0.0, 0.0, 0.0], 100.0, 100.0)
# hole = geom.add_disk([50.0, 50.0, 0.0], 20.0)
# geom.boolean_difference(plate, hole)
#     """
#     return system_instruction

# # これをLLMの「System Prompt」または「初期の指示文」として投げます
# print(generate_system_prompt())

In [116]:
rules_content = """# pygmsh.occ Geometry Generation Rules (Strict)

You are a CAD/CAE expert. Output ONLY valid Python code using the `pygmsh.occ` API.
The variable `geom` is already initialized.

### 1. Mandatory API Signatures:
- Rectangle: `rect = geom.add_rectangle([x0, y0, z0], width, height)`
- Disk (Circle): `disk = geom.add_disk([xc, yc, zc], radius)`
- Cylinder: `cyl = geom.add_cylinder([x0, y0, z0], [dx, dy, dz], radius)`
- Fillet (Edge rounding): `geom.add_fillet([line_entities], radius)`
- Boolean Union: `geom.boolean_union([entity1, entity2, ...])`
- Boolean Difference: `geom.boolean_difference(target_entity, tool_entity)`
- Boolean Intersection: `geom.boolean_intersection([entities])`

### 2. Physical Constraints:
- Use float values for all dimensions (e.g., 50.0).
- Ensure holes are completely contained within the base plate unless otherwise specified.
- Avoid perfectly overlapping edges from different operations to prevent mesh errors.

### 3. Output Format:
- No explanations, no markdown blocks (```), and no 'import' statements.
- Start directly with the geometry definitions.
- Assign the final resulting shape to a variable (e.g., `final_shape = ...`).

### 4. Physical Groups & Boundary Tags:
- To define boundaries for UNV/GMSH, use:
  `geom.add_physical(entity, label="MY_LABEL")`
- Example: `left_side = geom.add_rectangle(...)`, `geom.add_physical(left_side, "FIXED_SUPPORT")`
- This allows the UNV file to carry boundary condition information.

### 5. Essential for Export:
- When creating the final shape, always use `geom.add_physical(final_entity, "SURFACE")`.
- If the user specifies a boundary (e.g., "left edge"), create a curve and use `geom.add_physical(curve, "BC_NAME")`.
"""

# ファイルとして保存
with open("pygmsh_rules.txt", "w", encoding="utf-8") as f:
    f.write(rules_content)

print("✅ 'pygmsh_rules.txt' has been created successfully.")

✅ 'pygmsh_rules.txt' has been created successfully.


User Prompt: 添付した pygmsh_rules.txt のルールを厳守して、以下の形状を作成するPythonコードを出力してください。

要望： 縦300、横800の板の左端を固定したい。中央に三角形のような配置で3つの小さな穴（半径10）を開け、さらに右端の上下の角に小さな切り欠きを作って。

例：
1.
要望：縦200、横500の長方形板の右端を固定したい。中央付近に直径30の円孔を1つ開け、左下に幅40高さ20の長方形の切り欠きを作って。
2.
要望：縦400、横200の板の上下端を固定したい。板の中央に半径15の円孔を2つ、横並びで開け、右端中央に幅30高さ60の長方形の切り欠きを作って。
3.
要望：縦250、横600の板の左端を固定したい。板の中央に正三角形状に3つの穴（半径8）を開け、右上角に半径20の円弧状の切り欠きを作って。
4.
要望：縦100、横300の板の下端を固定したい。板の中央に直径20の円孔を1つ、左端上下に幅20高さ20の正方形の切り欠きを2つ作って。
5.
要望：縦350、横700の板の左端を固定したい。中央よりやや上に半径12の円孔を1つ、中央よりやや下に半径12の円孔を1つ開け、右端中央に幅50高さ30の長方形の切り欠きを作って。

In [ ]:
# LLMから出力されたコードをここに貼り付ける
# (実際にはAPI連携などで自動化も可能)
LLM_GEOMETRY_RECIPE = """
plate = geom.add_rectangle([0.0, 0.0, 0.0], 500.0, 200.0)
hole = geom.add_disk([250.0, 100.0, 0.0], 15.0)
notch = geom.add_rectangle([0.0, 0.0, 0.0], 40.0, 20.0)
plate_with_hole = geom.boolean_difference(plate, hole)
final_shape = geom.boolean_difference(plate_with_hole, notch)
geom.add_physical(final_shape, "SURFACE")
right_edge = geom.add_rectangle([500.0, 0.0, 0.0], 0.0, 200.0)
geom.add_physical(right_edge, "FIXED_SUPPORT")
"""

# def safe_mesh_execution(recipe, mesh_size=5.0):
#     import pygmsh
#     try:
#         with pygmsh.occ.Geometry() as geom:
#             geom.characteristic_length_min = mesh_size
#             geom.characteristic_length_max = mesh_size
            
#             # システムプロンプトで指示した通り、geom変数が存在する前提で実行
#             exec(recipe, {"geom": geom}) 
            
#             mesh = geom.generate_mesh()
#             return mesh
#     except Exception as e:
#         print(f"❌ LLMの生成コードにエラーがありました: {e}")
#         return None

# mesh = safe_mesh_execution(LLM_GEOMETRY_RECIPE)

def generate_mesh_from_llm(recipe_code, mesh_size=5.0):
    import pygmsh
    try:
        # 文字列から前後の空白やマークダウンのゴミを取り除く
        clean_code = recipe_code.strip().replace("```python", "").replace("```", "")
        
        with pygmsh.occ.Geometry() as geom:
            geom.characteristic_length_min = mesh_size
            geom.characteristic_length_max = mesh_size
            
            # 実行環境にgeomを渡す
            local_vars = {"geom": geom}
            exec(clean_code, {}, local_vars)
            
            mesh = geom.generate_mesh()
            return mesh
    except Exception as e:
        print(f"❌ Mesh generation failed: {e}")
        return None

In [ ]:
# ==========================================
# 📐 2D GEOMETRY RECIPE (Plate with holes)
# ==========================================
LLM_GEOMETRY_RECIPE_2D = """
plate = geom.add_rectangle([0.0, 0.0, 0.0], 500.0, 200.0)
hole = geom.add_disk([250.0, 100.0, 0.0], 15.0)
notch = geom.add_rectangle([0.0, 0.0, 0.0], 40.0, 20.0)
plate_with_hole = geom.boolean_difference(plate, hole)
final_shape = geom.boolean_difference(plate_with_hole, notch)
geom.add_physical(final_shape, "SURFACE")
right_edge = geom.add_rectangle([500.0, 0.0, 0.0], 0.0, 200.0)
geom.add_physical(right_edge, "FIXED_SUPPORT")
"""

def generate_mesh_2d(recipe_code, mesh_size=5.0):
    print("⏳ Generating 2D mesh...")
    try:
        clean_code = recipe_code.strip().replace("```python", "").replace("```", "")
        with pygmsh.occ.Geometry() as geom:
            geom.characteristic_length_min = mesh_size
            geom.characteristic_length_max = mesh_size
            exec(clean_code, {}, {"geom": geom})
            mesh = geom.generate_mesh()
        print(f"✅ 2D Mesh generated: {len(mesh.points)} nodes, {len(mesh.cells_dict.get('triangle', []))} triangles.")
        return mesh
    except Exception as e:
        print(f"❌ 2D Generation failed: {e}")
        return None

# 実行（2D）
mesh_2d = generate_mesh_2d(LLM_GEOMETRY_RECIPE_2D, mesh_size=20.0)

⏳ Generating 2D mesh...
✅ 2D Mesh generated: 1079 nodes, 2031 triangles.


In [ ]:
# ==========================================
# 🧊 3D GEOMETRY RECIPE (Solid Block with Hole)
# ==========================================
LLM_GEOMETRY_RECIPE_3D = """
plate = geom.add_rectangle([0.0, 0.0, 0.0], 500.0, 200.0)
hole = geom.add_disk([250.0, 100.0, 0.0], 15.0)
notch = geom.add_rectangle([0.0, 0.0, 0.0], 40.0, 20.0)
plate_with_hole = geom.boolean_difference(plate, hole)
final_shape = geom.boolean_difference(plate_with_hole, notch)
geom.add_physical(final_shape, "SURFACE")
right_edge = geom.add_rectangle([500.0, 0.0, 0.0], 0.0, 200.0)
geom.add_physical(right_edge, "FIXED_SUPPORT")
"""

def generate_mesh_3d(recipe_code, mesh_size=5.0):
    print("⏳ Generating 3D mesh...")
    try:
        clean_code = recipe_code.strip().replace("```python", "").replace("```", "")
        with pygmsh.occ.Geometry() as geom:
            geom.characteristic_length_min = mesh_size
            geom.characteristic_length_max = mesh_size
            exec(clean_code, {}, {"geom": geom})
            mesh = geom.generate_mesh()
        
        # 四面体要素があるか確認
        n_tetra = len(mesh.cells_dict.get('tetra', []))
        print(f"✅ 3D Mesh generated: {len(mesh.points)} nodes, {n_tetra} tetrahedrons.")
        return mesh
    except Exception as e:
        print(f"❌ 3D Generation failed: {e}")
        return None

# 実行（3D）
mesh_3d = generate_mesh_3d(LLM_GEOMETRY_RECIPE_3D, mesh_size=3.0)

⏳ Generating 3D mesh...
✅ 3D Mesh generated: 2418 nodes, 9023 tetrahedrons.


In [ ]:
import plotly.graph_objects as go

def visualize_mesh(mesh, title="Mesh"):
    if mesh is None: return

    points = mesh.points
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    
    # 3D (Tetra)
    if "tetra" in mesh.cells_dict:
        cells = mesh.cells_dict["tetra"]
        # 四面体を表面の三角形に変換して表示（簡易的）
        # ※本来は表面抽出が必要ですが、PlotlyのMesh3dは内部も含む全ごちゃ混ぜでも
        #   opacityを下げればそれっぽく見えます
        i, j, k = cells[:, 0], cells[:, 1], cells[:, 2] # 最初の3点だけ使う（簡易）
         # 正しくは4面すべてを描画すべきですが、データ量削減のためここでは割愛
        
        # より良い可視化：四面体の各面を三角形リストにする
        # [0,1,2], [0,2,3], [0,3,1], [1,3,2]
        tri_indices = np.vstack([
            cells[:, [0, 1, 2]], cells[:, [0, 2, 3]], 
            cells[:, [0, 3, 1]], cells[:, [1, 3, 2]]
        ])
        i, j, k = tri_indices[:, 0], tri_indices[:, 1], tri_indices[:, 2]
        
        fig = go.Figure(data=[go.Mesh3d(
            x=x, y=y, z=z, i=i, j=j, k=k, 
            opacity=0.3, color='cyan', name='3D Body'
        )])
        
    # 2D (Triangle)
    elif "triangle" in mesh.cells_dict:
        cells = mesh.cells_dict["triangle"]
        i, j, k = cells[:, 0], cells[:, 1], cells[:, 2]
        fig = go.Figure(data=[go.Mesh3d(
            x=x, y=y, z=z, i=i, j=j, k=k, 
            opacity=1.0, color='orange', name='2D Plate'
        )])
    
    else:
        print("⚠️ Unknown mesh type.")
        return

    fig.update_layout(title=title, scene=dict(aspectmode='data'))
    fig.show()

# 両方表示してみる
visualize_mesh(mesh_2d, "2D Mesh Preview")
visualize_mesh(mesh_3d, "3D Mesh Preview")

In [ ]:
# --- 5. 保存（形状データのみ） ---
if mesh is not None:
    nodes = mesh.points[:, :2]
    elements = mesh.cells_dict["triangle"]

    # 物理条件はここでは決めず、枠だけ作っておく
    input_data = {
        "nodes": nodes.tolist(),
        "elements": elements.tolist(),
        "material": {
            "E": 210000.0, 
            "thickness": 10.0
        },
        "conditions": {
            # 🌟 ここでは空にしておく（Pre側で判定させる）
            "fixed_nodes_x": [],
            "load_nodes": [],
            "total_force": 1000.0
        }
    }

    with open(OUTPUT_FILE, 'w') as f:
        json.dump(input_data, f, indent=4)
        
    print(f"🎉 Saved '{OUTPUT_FILE}' (Geometry only)")
    print(f"   Nodes: {len(nodes)}, Elements: {len(elements)}")
    print("   👉 Open the 'Pre' notebook to set boundary conditions.")

else:
    print("⚠️ Save skipped.")

🎉 Saved 'input.json' (Geometry only)
   Nodes: 11537, Elements: 22599
   👉 Open the 'Pre' notebook to set boundary conditions.


In [ ]:
%pip install --upgrade meshio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 🌟 ここでどちらを使うか選ぶ！
SELECTED_MESH = mesh_2d
# SELECTED_MESH = mesh_3d 

if SELECTED_MESH is not None:
    # 要素タイプの判定
    if "tetra" in SELECTED_MESH.cells_dict:
        e_type = "tetra"
        elements = SELECTED_MESH.cells_dict["tetra"].tolist()
    elif "triangle" in SELECTED_MESH.cells_dict:
        e_type = "triangle"
        elements = SELECTED_MESH.cells_dict["triangle"].tolist()
    else:
        e_type = "unknown"
        elements = []

    input_data = {
        "metadata": {"type": "3D" if e_type == "tetra" else "2D"},
        "nodes": SELECTED_MESH.points.tolist(),
        "elements": elements,
        "material": {"E": 210000.0, "nu": 0.3, "rho": 7.85e-9, "thickness": 10.0},
        "conditions": {"fixed_nodes_x": [], "load_nodes": [], "total_force": 1000.0},
        "simulation": {"num_steps": 100}
    }

    with open(OUTPUT_JSON_FILE, 'w') as f:
        json.dump(input_data, f, indent=4)
    
    print(f"🎉 Saved '{OUTPUT_JSON_FILE}' as {input_data['metadata']['type']} mesh.")
else:
    print("⚠️ No mesh selected.")

🎉 Saved 'input.json' as 2D mesh.
